# Cleaning multiple files in Python

This notebook cleans data from **one or several text files at once**.
The script is based on a tutorial by [Catris Code](https://catriscode.com/2021/05/01/tweets-cleaning-with-python/) and was first adapted and extended by Monika Barget (November 2022).

**Note:** This version runs entirely in your browser via Live Code (Pyodide). There is no access to a folder on your computer, so you upload your files below and download the cleaned versions at the end. Nothing is saved automatically, and everything is gone when you close the tab.

---

## Step 1: Import Required Libraries

We only need libraries for text cleaning, packing files into a zip archive, and talking to your browser (file upload). `nltk`, `requests` and `os` are not needed: Pyodide cannot run the downloads `nltk.download()` requires, and there is no local folder to list.

In [ ]:
import re
import io
import zipfile
import base64
from IPython.display import display, HTML
from js import document
from pyodide.ffi import create_proxy

## Step 2: Upload Your Files

Run the cell below. A **Choose files** button appears. Select one or more `.txt` files from your computer (one tweet per line). Hold **Ctrl** (Windows) or **Cmd** (Mac) to select several files in the file dialog.

If you want to add more files later, simply choose again: the new selection **replaces** the previous one.

**If no button appears or the upload fails**, use the paste option in the next cell instead.

In [ ]:
uploaded_files = {}   # filename -> full text of the file

display(HTML('<div id="tweet-upload-box"><input type="file" id="tweet-upload" accept=".txt" multiple> '
             '<span id="tweet-upload-status">No files selected yet.</span></div>'))

file_input = document.getElementById("tweet-upload")
status = document.getElementById("tweet-upload-status")

if file_input is None:
    # Fallback: add the controls to the page if the output was not yet in the DOM
    box = document.createElement("div")
    file_input = document.createElement("input")
    file_input.type = "file"
    file_input.accept = ".txt"
    file_input.multiple = True
    status = document.createElement("span")
    status.textContent = " No files selected yet."
    box.appendChild(file_input)
    box.appendChild(status)
    document.body.appendChild(box)

async def _read_files(event):
    uploaded_files.clear()
    n = file_input.files.length
    if n == 0:
        status.textContent = "No files selected yet."
        return
    for i in range(n):
        f = file_input.files.item(i)
        uploaded_files[f.name] = await f.text()
    names = ", ".join(uploaded_files.keys())
    status.textContent = f"Loaded {len(uploaded_files)} file(s): {names}. Continue with Step 3."

file_input.addEventListener("change", create_proxy(_read_files))

### Alternative: paste your tweets

Only use this cell if the upload above does not work. Paste your tweets between the triple quotes (one tweet per line), remove the `#` in front of the last two lines, and run the cell. You can then continue with Step 3 as normal.

In [ ]:
pasted_text = """
Paste your tweets here, one per line.
"""

# Uncomment the next two lines to use the pasted text instead of uploaded files
# uploaded_files = {"pasted_tweets.txt": pasted_text}
# print("Loaded pasted text as pasted_tweets.txt")

## Step 3: Define Custom Stopwords (optional)

Stopwords are common words (e.g., 'the', 'and') that can be filtered out. We use a simple manual list. Add your own words if you like.

In [ ]:
my_stopwords = ["http", "https", "#", "@"]
print("Custom stopwords:", my_stopwords)

## Step 4: Define the Cleaning Function

This function performs the following operations:
- Convert text to lowercase
- Expand common English contractions (so that "I'm" becomes "i am" and is not broken into "i" and "m")
- Remove mentions, URLs, punctuation, square brackets and special characters
- Filter out stopwords

In [ ]:
CONTRACTIONS = {
    "i'm": "i am", "we're": "we are", "you're": "you are", "she's": "she is",
    "he's": "he is", "they're": "they are", "don't": "do not", "won't": "will not",
    "it's": "it is",
}

def clean_tweet(tweet):
    if isinstance(tweet, float):  # covers NaN-like float values
        return " "
    temp = tweet.lower()
    temp = temp.replace("\u2019", "'")  # curly apostrophe -> straight apostrophe
    for short, full in CONTRACTIONS.items():  # avoid breaking English contractions
        temp = temp.replace(short, full)
    temp = re.sub(r'@[A-Za-z0-9_]+', ' ', temp)  # Remove mentions (e.g., @user)
    temp = re.sub("\n", " ", temp)
    temp = re.sub("\r", " ", temp)
    temp = re.sub(r'http\S+', ' ', temp)  # Remove URLs
    temp = re.sub('[()!?]', ' ', temp)  # Remove specific punctuation
    temp = re.sub(r'\[.*?\]', ' ', temp)  # Remove square brackets and content
    temp = re.sub("[^a-z0-9]", " ", temp)  # Keep only alphanumeric
    temp = temp.split()
    temp = [w for w in temp if w not in my_stopwords]  # Filter stopwords
    return " ".join(temp)

## Step 5: Clean All Uploaded Files

Each file is cleaned separately. The results are stored in a dictionary called `cleaned_files`, with the new file name as key. Names follow the pattern `originalname_cleaned-with-Python.txt`.

In [ ]:
cleaned_files = {}

if not uploaded_files:
    print("No files loaded yet. Go back to Step 2 and upload at least one file.")
else:
    for filename, text in uploaded_files.items():
        tweets = text.splitlines()
        results = [clean_tweet(tw) for tw in tweets]
        base_name = filename.rsplit(".", 1)[0]
        out_name = f"{base_name}_cleaned-with-Python.txt"
        cleaned_files[out_name] = "\n".join(results) + "\n"
        print(f"{filename}: {len(tweets)} lines -> {out_name}")
        for r in results[:3]:
            print("   -", r)
    print("done")

## Step 6: Download Your Results

Because this notebook runs inside your browser, there is no folder to write to. Running the cell below creates download links:
- one link for **all files together** as a `.zip` archive (recommended if you uploaded several files)
- one link **per cleaned file**

Click a link to save the file to your computer.

In [ ]:
if not cleaned_files:
    print("Nothing to download yet. Complete Steps 2 and 5 first.")
else:
    def _link(name, data_bytes, mime, label):
        b64 = base64.b64encode(data_bytes).decode("utf-8")
        return f'<p><a download="{name}" href="data:{mime};base64,{b64}" target="_blank">{label}</a></p>'

    html = ""

    # All files in one zip archive
    if len(cleaned_files) > 1:
        buffer = io.BytesIO()
        with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as zf:
            for name, content in cleaned_files.items():
                zf.writestr(name, content)
        html += _link("cleaned_tweets.zip", buffer.getvalue(), "application/zip",
                      f"Download all {len(cleaned_files)} files as cleaned_tweets.zip")

    # One link per file
    for name, content in cleaned_files.items():
        html += _link(name, content.encode("utf-8"), "text/plain", f"Download {name}")

    display(HTML(html))
    print("Cleaning complete. Click the links above to save your results.")